In [ ]:
### Drug Susceptibility Analysis with PharmacoGx. Generating gene signatures for drug sensitivity 
### and resistance using large public databases of drug response and transcriptomic data in cancer 
### cell lines using linear regression. Converting these gene signatures to Drosophila orthologs for
### mapping to single-cell RNA-seq data from Drosophila tumor models and performing gene ontology.

library(PharmacoGx)
library(curl)
library(dplyr)
library(pander)
library(biomaRt)
library(Seurat)
library(ggplot2)
library(cowplot)
library(patchwork)
library(gprofiler2)
library(clusterProfiler)
library(org.Dm.eg.db)
library(msigdbr)
library(tidyverse)

figDir <- "/data/ebaird/scRNAseq/SCENTINELsep24/GDSC/figures/"
tabDir <- "/data/ebaird/scRNAseq/SCENTINELsep24/GDSC/tables/"
dir.create(figDir, showWarnings = FALSE)
dir.create(tabDir, showWarnings = FALSE)

cols <- c(colorRamps::matlab.like2(20)[1:18], "deeppink2", "deeppink3", "deeppink4")

availablePSets()

In [ ]:
seu <- readRDS("/data/ebaird/scRNAseq/SCENTINELsep24/composition_DEG_signatures/signatures.rds")

In [ ]:
# 1. Download a pre-processed PharmacoSet (Pset).
# gdsc1 <- downloadPSet("GDSC_2020(v1-8.2)")
gdsc2 <- downloadPSet("GDSC_2020(v2-8.2)")
# ccle <- downloadPSet("CCLE_2015")
# GBM2 <- downloadPSet("GBM_scr2") 
# GBM3 <- downloadPSet("GBM_scr3") 
# gCSI <- downloadPSet("gCSI_2019")

gdsc2

In [ ]:
# See available drugs
drugNames(gdsc2)
# Search for a drug of interest
grep("temozolamide", drugNames(gdsc2), value = TRUE, ignore.case = TRUE)

In [ ]:
drug_name <- "Dinaciclib"  # Replace with your drug of interest
mDataType <- "rna"     # Molecular data type (e.g., RNA)
dataset <- gdsc2
dataset_name <- "gdsc2"
measure <- "aac_recomputed"  # Drug response measure (e.g., AUC, AAC, IC50)

In [ ]:
all_genes <- fNames(dataset, mDataType)

In [ ]:
sig_all <- drugSensitivitySig(
  object = dataset,
  mDataType = mDataType,
  drugs = drug_name,
  features = all_genes,
  sensitivity.measure = measure,
  molecular.summary.stat = "median",
  sensitivity.summary.stat = "median",
  verbose = FALSE
)

In [ ]:
# Convert 3D array to a data frame
results_df <- data.frame(
  Gene = dimnames(sig_all)[[1]],
  Estimate = sig_all[, 1, "estimate"],
  P_Value = sig_all[, 1, "pvalue"],
  FDR = sig_all[, 1, "fdr"],
  N = sig_all[, 1, "n"]
)

# Clean up Ensembl IDs by removing version numbers if they exist
results_df$Gene <- ifelse(
  grepl("\\.", results_df$Gene),      
  sub("\\..*", "", results_df$Gene),  
  results_df$Gene                     
)

# Filter for significant associations (FDR < 0.05)
results_df <- results_df %>%
  filter(FDR < 0.05) %>%
  arrange(FDR)

head(results_df)

In [ ]:
# Convert ENSEMBL IDs to human Gene Symbols
ensembl <- useMart("ensembl", dataset = "hsapiens_gene_ensembl")
gene_mapping <- getBM(
  attributes = c("ensembl_gene_id", "hgnc_symbol"),
  filters = "ensembl_gene_id",
  values = results_df$Gene,
  mart = ensembl
)
results_df_human <- results_df %>%
  dplyr::left_join(gene_mapping, by = c("Gene" = "ensembl_gene_id")) %>%
  dplyr::select(Gene = hgnc_symbol, Estimate, P_Value, FDR, N) %>%
  dplyr::filter(!is.na(Gene))

# Number of genes with no symbol mapping
num_no_symbol <- sum(is.na(results_df_human$Gene))
cat("Number of genes with no symbol mapping:", num_no_symbol, "\n")

In [ ]:
# Select only positive estimates and sort by p-value and select top 100 genes
top_100_pos_human <- results_df_human[results_df_human$Estimate > 0, ] |>
  dplyr::arrange(P_Value) |>
  dplyr::slice_head(n = 100)

top_100_neg_human <- results_df_human[results_df_human$Estimate < 0, ] |>
  dplyr::arrange(P_Value) |>
  dplyr::slice_head(n = 100)

In [ ]:
pander::pandoc.table(
  top_100_pos_human,
  style = "rmarkdown",
  caption = paste("Top 100 Genes Associated with sensitivity to", drug_name)
)

pander::pandoc.table(
  top_100_neg_human,
  style = "rmarkdown",
  caption = paste("Top 100 Genes Associated with resistance to", drug_name)
)

In [ ]:
# Get orthologs using g:Profiler
orthologs <- gorth(
  query = results_df$Gene,
  source_organism = "hsapiens",
  target_organism = "dmelanogaster",
  numeric_ns = "ENSEMBL"
)

# Process results
results_df_drosophila <- results_df %>%
  left_join(orthologs, by = c("Gene" = "input_ensg")) %>%
  dplyr::select(Drosophila_Gene = ortholog_name, 
         Human_Gene = input, 
         Human_Ensembl = Gene,
         Estimate, P_Value, FDR, N) %>%
  filter(!is.na(Drosophila_Gene)) %>%
  distinct(Drosophila_Gene, .keep_all = TRUE)

# Number of genes with no orthologs as variable
num_genes_no_ortholog <- length(results_df$Gene) - length(results_df_drosophila$Drosophila_Gene)

# Remove genes not in seu
results_df_drosophila <- results_df_drosophila %>%
  filter(Drosophila_Gene %in% rownames(seu))

# Number of genes not in seu as variable
num_genes_not_in_seu <- length(results_df$Gene) - length(results_df_drosophila$Drosophila_Gene)

# View the results
print(head(results_df_drosophila))

In [ ]:
# Select only positive estimates and sort by p-value and select top 100 genes
top_100_pos_fly <- results_df_drosophila[results_df_drosophila$Estimate > 0, ] |>
  dplyr::arrange(P_Value) |>
  dplyr::slice_head(n = 100)

top_100_neg_fly <- results_df_drosophila[results_df_drosophila$Estimate < 0, ] |>
  dplyr::arrange(P_Value) |>
  dplyr::slice_head(n = 100)

In [ ]:
pander::pandoc.table(
  top_100_pos_fly,
  style = "rmarkdown",
  caption = paste("Top 100 Genes Associated with sensitivity to", drug_name)
)

pander::pandoc.table(
  top_100_neg_fly,
  style = "rmarkdown",
  caption = paste("Top 100 Genes Associated with resistance to", drug_name)
)

In [ ]:
# Save results to CSV
write.csv(top_100_pos_fly, file = paste0(tabDir, dataset_name, measure, drug_name, "_top100_sensitivity.csv"), row.names = FALSE)
write.csv(top_100_neg_fly, file = paste0(tabDir, dataset_name, measure, drug_name, "_top100_resistance.csv"), row.names = FALSE)

In [ ]:
# Top 100 pos fly gene lists
top_100_fly_genes_pos <- top_100_pos_fly$Drosophila_Gene
top_100_fly_genes_neg <- top_100_neg_fly$Drosophila_Gene

head(top_100_fly_genes_pos)
head(top_100_fly_genes_neg)

In [ ]:
### Add module score for top 100 pos and top 100 neg
seu <- AddModuleScore(seu, features = list(top_100_fly_genes_pos), name = paste0(dataset_name, measure, drug_name, "_top100_sensitivity_genes"))
seu <- AddModuleScore(seu, features = list(top_100_fly_genes_neg), name = paste0(dataset_name, measure, drug_name, "_top100_resistance_genes"))

# Visualize the module scores on UMAP
p <- FeaturePlot(seu, features = paste0(dataset_name, measure, drug_name, "_top100_sensitivity_genes1"), order = TRUE, ncol = 1) & 
  scale_colour_gradientn(colours = cols) & 
  ggtitle(paste0(drug_name, " Top 100 Sensitivity genes Score"))

q <- FeaturePlot(seu, features = paste0(dataset_name, measure, drug_name, "_top100_resistance_genes1"), order = TRUE, ncol = 1) & 
  scale_colour_gradientn(colours = cols) & 
  ggtitle(paste0(drug_name, " Top 100 Resistance genes Score"))

# Save plots
jpeg_file <- paste0(figDir, "UMAP.", dataset_name, measure, drug_name, ".jpeg")
jpeg(filename = jpeg_file, width = 1000, height = 2000, res = 150)
gridExtra::grid.arrange(p, q, ncol = 1)
dev.off()

In [ ]:
### Create a report
# Include: number of genes used in AddModuleScore (top100_fly_genes_pos present in seu), average estimate,
# , median estimate, min and max estimate for pos and neg genes, average FDR, average p-value etc.

# Report
report <- data.frame(
  Metric = c("Number of Pos Genes in Seurat", "Number of Neg Genes in Seurat", 
             "Number Genes not in Seurat", "Number of Genes with No Ortholog", "Number of genes with No Symbol",
             "Average Estimate Pos", "Median Estimate Pos", "Min Estimate Pos", "Max Estimate Pos",
             "Average FDR Pos", "Average P-Value Pos",
             "Average Estimate Neg", "Median Estimate Neg", "Min Estimate Neg", "Max Estimate Neg",
             "Average FDR Neg", "Average P-Value Neg"),
  Value = c(
    length(top_100_fly_genes_pos),
    length(top_100_fly_genes_neg),
    num_genes_not_in_seu,
    num_genes_no_ortholog,
    num_no_symbol,
    mean(top_100_pos_fly$Estimate, na.rm = TRUE),
    median(top_100_pos_fly$Estimate, na.rm = TRUE),
    min(top_100_pos_fly$Estimate, na.rm = TRUE),
    max(top_100_pos_fly$Estimate, na.rm = TRUE),
    mean(top_100_pos_fly$FDR, na.rm = TRUE),
    mean(top_100_pos_fly$P_Value, na.rm = TRUE),
    mean(top_100_neg_fly$Estimate, na.rm = TRUE),
    median(top_100_neg_fly$Estimate, na.rm = TRUE),
    min(top_100_neg_fly$Estimate, na.rm = TRUE),
    max(top_100_neg_fly$Estimate, na.rm = TRUE),
    mean(top_100_neg_fly$FDR, na.rm = TRUE),
    mean(top_100_neg_fly$P_Value, na.rm = TRUE)
  )
)

pander::pandoc.table(report, style = "rmarkdown", caption = paste("Report for", drug_name, "Gene Signatures"))

# Save report to CSV
write.csv(report, file = paste0(tabDir, dataset_name, measure, drug_name, "_gene_signature_report.csv"), row.names = FALSE)

In [ ]:
# Gene ontology for top 100 pos and neg genes

ego_pos <- enrichGO(
  gene = top_100_fly_genes_pos,
  OrgDb = org.Dm.eg.db,
  keyType = "SYMBOL",
  ont = "BP",
  pAdjustMethod = "BH",
  pvalueCutoff = 0.05,
  qvalueCutoff = 0.2,
  readable = TRUE
)

ego_neg <- enrichGO(
  gene = top_100_fly_genes_neg,
  OrgDb = org.Dm.eg.db,
  keyType = "SYMBOL",
  ont = "BP",
  pAdjustMethod = "BH",
  pvalueCutoff = 0.05,
  qvalueCutoff = 0.2,
  readable = TRUE
)   

# Save plots
figDir <- "/data/ebaird/scRNAseq/SCENTINELsep24/GDSC/"  
jpeg_file <- paste0(figDir, "GO_Top100_", dataset_name, measure, drug_name, "_sensitivity_resistance.jpeg")
jpeg(filename = jpeg_file, width = 2000, height = 1500, res = 150)
gridExtra::grid.arrange(
  dotplot(ego_pos, showCategory = 10) + ggtitle("GO Biological Processes - Top 100 Sensitivity Genes"),
  dotplot(ego_neg, showCategory = 10) + ggtitle("GO Biological Processes - Top 100 Resistance Genes"),
  ncol = 1
)
dev.off()